# Serverless BNG mini-COG mosaic validation

Proves the **light-tier `gridSystem="bng"`** write → read works end-to-end
on real Serverless (e2-demo / oauth-fe).

A native EPSG:27700 synthetic raster (200×200 @ 100 m/px, near London,
E=530 000 N=180 000) is written, split into one mini-COG per overlapping
BNG 1-km cell via `cog_gbx`, then re-expanded with `raster_gbx`. Every
cell's `cellid` (BNG string), `gridSystem`, CRS (EPSG:27700), and `rst_avg`
are validated before the notebook exits with a structured result dict.

In [ ]:
import datetime
import json
import os
import shutil

# ---- config ----------------------------------------------------------------
VOL_BASE = "/Volumes/geospatial_docs/geobrix/sample-data"
TIMESTAMP = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = f"{VOL_BASE}/bng-mosaic-validation/{TIMESTAMP}"
SRC_DIR = f"{RUN_DIR}/source"
OUT_DIR = f"{RUN_DIR}/mosaic_bng"
GRID_RESOLUTION = "1km"
GRID_SYSTEM = "bng"

os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print("run_dir :", RUN_DIR)

results = {
    "run_dir": RUN_DIR,
    "grid_system": GRID_SYSTEM,
    "grid_resolution": GRID_RESOLUTION,
    "stages": {},
}

In [ ]:
# Write a 200x200, 100 m/px, EPSG:27700 synthetic raster near London.
# Upper-left at E=530 000, N=180 000 — within the GB EPSG:27700 envelope.
# Native 27700 source avoids any datum-shift grid dependency (OSTN15 absent on Serverless).
import numpy as np
import rasterio
from rasterio.transform import from_origin

TMP_SRC = "/tmp/bng_val_source_gb.tif"
w, h = 200, 200
data = np.arange(w * h, dtype=np.uint16).reshape(1, h, w) % 60000
profile = dict(
    driver="GTiff",
    width=w,
    height=h,
    count=1,
    dtype="uint16",
    crs="EPSG:27700",
    transform=from_origin(530000.0, 180000.0, 100.0, 100.0),
)
with rasterio.open(TMP_SRC, "w", **profile) as ds:
    ds.write(data)

VOL_SRC = f"{SRC_DIR}/source_gb.tif"
shutil.copy(TMP_SRC, VOL_SRC)
print("source:", VOL_SRC, os.path.getsize(VOL_SRC), "bytes")
results["source_path"] = VOL_SRC

In [ ]:
# Register light datasources, then write the BNG grid mosaic.
from databricks.labs.gbx.ds.register import register

register(spark)

(
    spark.read.format("file_gbx")
    .load(VOL_SRC)
    .write.format("cog_gbx")
    .option("gridSystem", GRID_SYSTEM)
    .option("gridResolution", GRID_RESOLUTION)
    .mode("overwrite")
    .save(OUT_DIR)
)

MOSAIC_VRT = f"{OUT_DIR}/mosaic.vrt"
assert os.path.exists(MOSAIC_VRT), f"bng mosaic mode must write mosaic.vrt at {MOSAIC_VRT}"

cell_tiles = sorted(
    os.path.join(OUT_DIR, f)
    for f in os.listdir(OUT_DIR)
    if f.startswith("cell_") and f.lower().endswith(".tif")
)
assert len(cell_tiles) >= 1, f"bng mosaic must produce at least one cell mini-COG, got {len(cell_tiles)}"
print(f"mosaic.vrt written; {len(cell_tiles)} cell mini-COGs")
results["stages"]["write"] = {"n_cell_tiles": len(cell_tiles)}

In [ ]:
# Verify CRS of first cell tile on driver, then expand via raster_gbx.
from pyspark.sql.functions import col

from databricks.labs.gbx.pyrx.functions import rst_avg

with rasterio.open(cell_tiles[0]) as ds:
    cell_epsg = ds.crs.to_epsg()
print(f"first cell CRS: EPSG:{cell_epsg}")
assert cell_epsg == 27700, f"bng cell must be in EPSG:27700, got EPSG:{cell_epsg}"

df = spark.read.format("raster_gbx").load(MOSAIC_VRT)
n_rows = df.count()
print(f"raster_gbx rows: {n_rows} (expected {len(cell_tiles)} cell tiles)")
assert n_rows == len(cell_tiles), f"row count {n_rows} != cell tile count {len(cell_tiles)}"

rows = df.select(
    col("tile.path").alias("path"),
    col("tile.metadata")["cellid"].alias("cellid"),
    col("tile.metadata")["gridSystem"].alias("grid_system"),
    rst_avg(col("tile")).alias("avg"),
).collect()

for r in rows:
    print(f"  cellid={r['cellid']!r}  grid_system={r['grid_system']!r}  avg={r['avg']}")

results["stages"]["read"] = {
    "n_rows": n_rows,
    "sample_cellid": rows[0]["cellid"] if rows else None,
    "cell_epsg": cell_epsg,
}

In [ ]:
# Assert all invariants, build final results dict, exit.
bad_cellid = [r for r in rows if not r["cellid"]]
bad_gs = [r for r in rows if r["grid_system"] != "bng"]
null_avg = [r for r in rows if r["avg"] is None]

assert not bad_cellid, f"{len(bad_cellid)} rows have empty cellid"
assert not bad_gs, f"{len(bad_gs)} rows have wrong gridSystem (expected 'bng')"
assert len(null_avg) < len(rows), f"all {len(rows)} rows have null rst_avg"

results["n_rows"] = n_rows
results["n_cell_tiles"] = len(cell_tiles)
results["sample_cellid"] = rows[0]["cellid"] if rows else None
results["null_avg_rows"] = len(null_avg)
results["pass"] = True

print(json.dumps(results, indent=2))
print("\nPASS: BNG mini-COG mosaic write \u2192 read validated on Serverless.")
dbutils.notebook.exit(json.dumps(results))